# Visual Classifier — Flexible Fine-Tuning Workbench

This notebook is a **flexible experimentation workbench** for fine-tuning
the `dima806/ai_vs_human_generated_image_detection` ViT model on Apple Silicon.

### How to use

1. **Run sections 0–2** once to set up the environment and load datasets.
2. **Edit the config block** in Section 3, then run the training cell.
3. **Evaluate** in Section 4.
4. **Repeat** — change the config and re-run Section 3 as many times as you want.
   - Swap the training order (julienlucas first, genimage second)
   - Re-run with different hyperparameters
   - Continue from a saved checkpoint or from the base model
   - Add replay from a previous stage

| Dataset | Key | Purpose |
|---------|-----|--------|
| `nebula/GenImage-arrow` | `genimage` | Broaden the model on diverse AI generators |
| `julienlucas/midjourney-dalle-sd-…` | `julienlucas` | Specialise on Midjourney / DALL·E / SD |

The task is **binary classification**: `0 = Real`, `1 = AI-Generated`.

> ⚠️ **Memory:** The M4 has unified memory (16–32 GB). Batch sizes are
> kept small to avoid OOM. Adjust `BATCH_SIZE` if you have more RAM.

> ⚠️ **Large model checkpoints are .gitignored.** Only lightweight deltas or adapters should be committed.

---
## 0 · Shared Configuration
Global constants shared by all cells.  Edit once at the start.

In [1]:
# ── Base model ──
BASE_MODEL = "dima806/ai_vs_human_generated_image_detection"

# ── GenImage streaming subset sizes ──
TRAIN_PER_CLASS = 4800
VAL_PER_CLASS   = 550
TEST_PER_CLASS  = 1000
SEED            = 42

# ── Output paths ──
MODELS_DIR      = "outputs/models"
EVAL_OUTPUT_DIR  = "outputs"

---
## 1 · Environment Setup

In [2]:
import os, sys, torch

# ── GPU / Accelerator check ──
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("✅ MPS (Apple Silicon) available")
    USE_FP16 = False   # fp16 not fully supported on MPS
elif torch.cuda.is_available():
    print(f"✅ CUDA available – {torch.cuda.get_device_name(0)}")
    USE_FP16 = True
else:
    print("⚠️  No GPU – training will be slow on CPU")
    USE_FP16 = False

print(f"Mixed precision (fp16): {USE_FP16}")
print(f"PyTorch version: {torch.__version__}")

✅ MPS (Apple Silicon) available
Mixed precision (fp16): False
PyTorch version: 2.11.0


### Authentication

GenImage-arrow may require a Hugging Face token.

**Do NOT hardcode your token.** Use one of these safe methods:
- Set the `HF_TOKEN` environment variable before launching the notebook.
- Or run `huggingface_hub.login()` / `notebook_login()` interactively.

In [3]:
HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN is None:
    try:
        from huggingface_hub import notebook_login
        notebook_login()          # interactive widget / prompt
        HF_TOKEN = True           # login() stores the token globally
    except Exception:
        print("⚠️  No HF token found. Public datasets will still work,"
              " but gated datasets may fail.")
        HF_TOKEN = None
else:
    print("✅ HF_TOKEN loaded from environment.")

### Imports

In [4]:
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoImageProcessor, AutoModelForImageClassification

# Our helper module (lives next to this notebook)
from two_stage_finetuning import (
    get_device,
    load_model_from,
    build_genimage_subset,
    load_julienlucas_dataset,
    run_training_stage,
    evaluate_model,
)
# Weight-delta helpers from the existing visual_classifier module
from visual_classifier import save_weight_delta

DEVICE = get_device()
print(f"Device: {DEVICE}")

Device: mps


---
## 2 · Load Datasets

Run each cell **once** to load the dataset into memory.  
You only need to load the datasets you plan to use — skip the ones you don't need.

### 2a · GenImage (streamed subset)

In [6]:
genimage_ds = build_genimage_subset(
    train_per_class=TRAIN_PER_CLASS,
    val_per_class=VAL_PER_CLASS,
    test_per_class=TEST_PER_CLASS,
    seed=SEED,
    hf_token=HF_TOKEN if isinstance(HF_TOKEN, str) else None,
)
genimage_ds

📡  Streaming GenImage-arrow (this may take a few minutes)…


Resolving data files:   0%|          | 0/1216 [00:00<?, ?it/s]

   Columns: ['image_path', 'md5', 'width', 'height', 'image']
   Example image_path: ADM/train/nature/n02966687_11178.JPEG


Resolving data files:   0%|          | 0/1216 [00:00<?, ?it/s]

   collected 500/12700…
   collected 1000/12700…
   collected 1500/12700…
   collected 2000/12700…
   collected 2500/12700…
   collected 3000/12700…
   collected 3500/12700…
   collected 4000/12700…
   collected 4500/12700…
   collected 5000/12700…
   collected 5500/12700…
   collected 6000/12700…
   collected 6500/12700…
   collected 7000/12700…
   collected 7500/12700…
   collected 8000/12700…
   collected 8500/12700…
   collected 9000/12700…
   collected 9500/12700…
   collected 10000/12700…
   collected 10500/12700…
   collected 11000/12700…
   collected 11500/12700…
   collected 12000/12700…
   collected 12500/12700…
   train: 9600 samples (real=4800, ai=4800)
   validation: 1100 samples (real=550, ai=550)
   test: 2000 samples (real=1000, ai=1000)


DatasetDict({
    train: Dataset({
        features: ['image_path', 'md5', 'width', 'height', 'image', 'label'],
        num_rows: 9600
    })
    validation: Dataset({
        features: ['image_path', 'md5', 'width', 'height', 'image', 'label'],
        num_rows: 1100
    })
    test: Dataset({
        features: ['image_path', 'md5', 'width', 'height', 'image', 'label'],
        num_rows: 2000
    })
})

### 2b · julienlucas dataset

In [7]:
julienlucas_ds = load_julienlucas_dataset(
    val_ratio=0.1,
    seed=SEED,
    hf_token=HF_TOKEN if isinstance(HF_TOKEN, str) else None,
)
julienlucas_ds

✅  julienlucas dataset loaded  –  splits: ['train', 'test']
   train: 10695 samples
   test: 2000 samples
   → created validation split (1070 samples)
   Label column present: True
   Sample label value : 0


DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 9625
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 1070
    })
    test: Dataset({
        features: ['image', 'label'],
        num_rows: 2000
    })
})

---
## 3 · 🔧 Training Run

**Edit the config block below**, then run the cell.  
Re-run this cell as many times as you want with different settings.

#### `START_FROM` options
| Value | Meaning |
|-------|---------|
| `"base"` | Fresh start from the original dima806 model |
| `"outputs/models/run_01_genimage"` | Continue from a previously saved checkpoint |
| `"current"` | Continue from whatever model was trained in the last run (still in memory) |

#### `TRAIN_DATASET` / `REPLAY_DATASET` options
| Value | Dataset |
|-------|---------|
| `"genimage"` | GenImage streaming subset (loaded in 2a) |
| `"julienlucas"` | julienlucas Midjourney/DALL·E/SD (loaded in 2b) |

In [20]:
# ═══════════════════════════════════════════════════════════════
#  🔧  TRAINING RUN CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
RUN_NAME         = "run_02_ft_genimage_w_julienlucas"
START_FROM       = "outputs/models/run_02_ft_genimage"
TRAIN_DATASET    = "julienlucas"
REPLAY_DATASET   = "genimage"
EPOCHS           = 4
BATCH_SIZE       = 6
LEARNING_RATE    = 3e-5
FREEZE_LAYERS    = 10
AUGMENT          = True
WEIGHT_DECAY     = 0.05
EARLY_STOPPING   = 2
REPLAY_RATIO     = 0.25

In [21]:
# ── Resolve starting model ──
if START_FROM == "current":
    assert 'current_model' in dir() and current_model is not None, \
        "No current_model in memory. Use 'base' or a saved checkpoint path."
    model = current_model
    print(f"📦  Continuing from current in-memory model")
else:
    model, processor = load_model_from(source=START_FROM, device=DEVICE)

# ── Resolve datasets ──
_datasets = {
    "genimage":    genimage_ds    if 'genimage_ds'    in dir() else None,
    "julienlucas": julienlucas_ds if 'julienlucas_ds' in dir() else None,
}

train_data = _datasets[TRAIN_DATASET]
assert train_data is not None, f"Dataset '{TRAIN_DATASET}' not loaded. Run the loader cell first."

replay_data = None
if REPLAY_DATASET is not None:
    replay_data = _datasets[REPLAY_DATASET]
    assert replay_data is not None, f"Replay dataset '{REPLAY_DATASET}' not loaded. Run its loader cell first."
    replay_data = replay_data["train"]

# ── Output directory ──
run_output_dir = os.path.join(MODELS_DIR, RUN_NAME)

# ── Print summary ──
print(f"\n{'─'*50}")
print(f"  Run:           {RUN_NAME}")
print(f"  Start from:    {START_FROM}")
print(f"  Train on:      {TRAIN_DATASET} ({len(train_data['train'])} train samples)")
print(f"  Replay from:   {REPLAY_DATASET or 'None'}")
print(f"  Epochs:        {EPOCHS}")
print(f"  Batch size:    {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Freeze layers: {FREEZE_LAYERS or 'None (full fine-tune)'}")
print(f"  Augment:       {AUGMENT}")
print(f"  Output:        {run_output_dir}")
print(f"{'─'*50}\n")

# ── Train ──
current_model, current_trainer = run_training_stage(
    model=model,
    processor=processor,
    train_ds=train_data["train"],
    eval_ds=train_data["validation"],
    output_dir=run_output_dir,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    stage_name=RUN_NAME,
    fp16=USE_FP16,
    replay_ds=replay_data,
    replay_ratio=REPLAY_RATIO,
    freeze_layers=FREEZE_LAYERS,
    augment=AUGMENT,
    weight_decay=WEIGHT_DECAY,
    early_stopping_patience=EARLY_STOPPING,
)

print(f"\n✅  Training complete.  Model stored in `current_model`.")
print(f"    Saved to: {run_output_dir}")
print(f"    To continue training from this model, set START_FROM = 'current' or '{run_output_dir}'")

📦  Loading model from: outputs/models/run_02_ft_genimage


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

✅  Model loaded  –  85,800,194 parameters  (device: mps)

──────────────────────────────────────────────────
  Run:           run_02_ft_genimage_w_julienlucas
  Start from:    outputs/models/run_02_ft_genimage
  Train on:      julienlucas (9625 train samples)
  Replay from:   genimage
  Epochs:        4
  Batch size:    6
  Learning rate: 3e-05
  Freeze layers: 10
  Augment:       True
  Output:        outputs/models/run_02_ft_genimage_w_julienlucas
──────────────────────────────────────────────────


  🚀  run_02_ft_genimage_w_julienlucas


Filter:   0%|          | 0/9600 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9600 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** train metrics *****
  epoch                    =          4.0
  total_flos               = 3473110970GF
  train_loss               =       1.9511
  train_runtime            =   0:58:18.00
  train_samples_per_second =       13.758
  train_steps_per_second   =        0.574
  ✅  Model saved → outputs/models/run_02_ft_genimage_w_julienlucas

✅  Training complete.  Model stored in `current_model`.
    Saved to: outputs/models/run_02_ft_genimage_w_julienlucas
    To continue training from this model, set START_FROM = 'current' or 'outputs/models/run_02_ft_genimage_w_julienlucas'


---
## 4 · 📊 Evaluate Current Model

Evaluates `current_model` on whichever test set(s) you choose.  
Edit the config, then run.

In [22]:
# ═══════════════════════════════════════════════════════════════
#  📊  EVALUATION CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
EVAL_ON     = ["genimage", "julienlucas"]   # which test sets to evaluate on
EVAL_PREFIX = RUN_NAME                      # prefix for output files
EVAL_BATCH  = 6                             # batch size for evaluation

In [23]:
assert 'current_model' in dir() and current_model is not None, \
    "No current_model to evaluate. Run a training cell first."

_test_sets = {
    "genimage":    genimage_ds["test"]    if 'genimage_ds'    in dir() else None,
    "julienlucas": julienlucas_ds["test"] if 'julienlucas_ds' in dir() else None,
}

eval_results = {}
for ds_name in EVAL_ON:
    test_ds = _test_sets.get(ds_name)
    if test_ds is None:
        print(f"⚠️  Skipping {ds_name} — dataset not loaded")
        continue
    print(f"\n{'='*50}")
    print(f"  Evaluating on: {ds_name} ({len(test_ds)} samples)")
    print(f"{'='*50}")

    prefix = f"{EVAL_PREFIX}_on_ds_{ds_name}"
    metrics = evaluate_model(
        model=current_model,
        processor=processor,
        test_ds=test_ds,
        output_prefix=prefix,
        output_dir=EVAL_OUTPUT_DIR,
        batch_size=EVAL_BATCH,
        fp16=USE_FP16,
    )
    eval_results[ds_name] = metrics

# ── Quick comparison table ──
if eval_results:
    print(f"\n{'='*60}")
    print(f"  📋  Summary for: {EVAL_PREFIX}")
    print(f"{'='*60}")
    print(f"  {'Dataset':<15} {'Accuracy':>10} {'F1':>10} {'Precision':>10} {'Recall':>10}")
    print(f"  {'─'*55}")
    for name, m in eval_results.items():
        print(f"  {name:<15} {m['accuracy']:>10.4f} {m['f1']:>10.4f} {m['precision']:>10.4f} {m['recall']:>10.4f}")


  Evaluating on: genimage (2000 samples)


  📄  Metrics saved → outputs/run_02_ft_genimage_w_julienlucas_on_ds_genimage_eval_results.json
  🖼️   Confusion matrix saved → outputs/run_02_ft_genimage_w_julienlucas_on_ds_genimage_confusion_matrix.png

  ────────────────────────────────────────
  Accuracy  : 0.7750
  Precision : 0.7505
  Recall    : 0.8240
  F1-score  : 0.7855
  ────────────────────────────────────────

              precision    recall  f1-score   support

        Real       0.80      0.73      0.76      1000
AI-Generated       0.75      0.82      0.79      1000

    accuracy                           0.78      2000
   macro avg       0.78      0.77      0.77      2000
weighted avg       0.78      0.78      0.77      2000


  Evaluating on: julienlucas (2000 samples)


  📄  Metrics saved → outputs/run_02_ft_genimage_w_julienlucas_on_ds_julienlucas_eval_results.json
  🖼️   Confusion matrix saved → outputs/run_02_ft_genimage_w_julienlucas_on_ds_julienlucas_confusion_matrix.png

  ────────────────────────────────────────
  Accuracy  : 0.8200
  Precision : 0.8053
  Recall    : 0.8440
  F1-score  : 0.8242
  ────────────────────────────────────────

              precision    recall  f1-score   support

        Real       0.84      0.80      0.82      1000
AI-Generated       0.81      0.84      0.82      1000

    accuracy                           0.82      2000
   macro avg       0.82      0.82      0.82      2000
weighted avg       0.82      0.82      0.82      2000


  📋  Summary for: run_02_ft_genimage_w_julienlucas
  Dataset           Accuracy         F1  Precision     Recall
  ───────────────────────────────────────────────────────
  genimage            0.7750     0.7855     0.7505     0.8240
  julienlucas         0.8200     0.8242     0.8053     0.

---
## 5 · 💾 Save Weight Delta

Save only the **weight differences** from the base model.  
The resulting file is typically ~74 MB (int8 quantised) and stays under GitHub's 100 MB limit.

> This is **full fine-tuning**, not LoRA/PEFT. The delta is not a
> lightweight adapter — it encodes changes across all parameters.

In [26]:
# ═══════════════════════════════════════════════════════════════
#  💾  DELTA CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
DELTA_OUTPUT_PATH = f"./fine_tuned_model_delta/{RUN_NAME}_weight_delta.pt"

In [27]:
# Reload module to pick up any fixes
import importlib, visual_classifier
importlib.reload(visual_classifier)
from visual_classifier import save_weight_delta

assert 'current_model' in dir() and current_model is not None, \
    "No current_model to save. Run a training cell first."

delta_path, delta_mb = save_weight_delta(
    fine_tuned_model=current_model,
    base_model_name=BASE_MODEL,
    output_path=DELTA_OUTPUT_PATH,
)
print(f"\n✅ Delta saved to '{delta_path}' ({delta_mb:.2f} MB)")

Loading base model 'dima806/ai_vs_human_generated_image_detection' to compute delta...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Delta saved → './fine_tuned_model_delta/run_02_ft_genimage_w_julienlucas_weight_delta.pt'
  Changed parameters : 68
  Unchanged (skipped): 132
  File size          : 27.07 MB

✅ Delta saved to './fine_tuned_model_delta/run_02_ft_genimage_w_julienlucas_weight_delta.pt' (27.07 MB)


---
## 6 · 🔍 Compare Models on Sample Images

Load any combination of saved models and compare predictions side-by-side.

In [28]:
# ═══════════════════════════════════════════════════════════════
#  🔍  COMPARISON CONFIG — edit this block, then run the cell
# ═══════════════════════════════════════════════════════════════
# Each entry: (display_name, model_name_or_path, delta_path_or_None)
MODELS_TO_COMPARE = [
    ("Base",                        "dima806/ai_vs_human_generated_image_detection",    None),
    ("GenImage",                    "outputs/models/run_01_ft_genimage",                None),
    ("GenImage then JulienLucas",   "outputs/models/run_02_ft_genimage_w_julienlucas",  None),
    # ("Delta",      "dima806/ai_vs_human_generated_image_detection", "fine_tuned_model_delta/weight_delta.pt"),
]

SAMPLE_DIR = "../../data/sample_images"

In [29]:
from visual_classifier import VisualClassifier
from PIL import Image

# ── Load all comparison models ──
loaded_models = []
for display_name, model_path, delta in MODELS_TO_COMPARE:
    print(f"Loading: {display_name} ...")
    vc = VisualClassifier(
        model_name_or_path=model_path,
        delta_path=delta,
    )
    loaded_models.append((display_name, vc))

print(f"\n✅  Loaded {len(loaded_models)} models for comparison\n")

# ── Run predictions on sample images ──
for fname in sorted(os.listdir(SAMPLE_DIR)):
    if fname.startswith("."):
        continue
    img = Image.open(os.path.join(SAMPLE_DIR, fname))
    expected = "Real" if fname.startswith("real") else "AI Generated"

    print(f"\n{fname}")
    print(f"  Expected: {expected}")
    for display_name, vc in loaded_models:
        result = vc.predict(img)
        marker = "✅" if result['prediction'] == expected else "❌"
        print(f"  {marker} {display_name:>15}:  {result['prediction']} ({result['confidence']:.2%})")

Loading: Base ...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading: GenImage ...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading: GenImage then JulienLucas ...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]


✅  Loaded 3 models for comparison


fake+metadata+place-face.png
  Expected: AI Generated
  ❌            Base:  Real (99.06%)
  ❌        GenImage:  Real (69.15%)
  ✅ GenImage then JulienLucas:  AI Generated (66.36%)

fake+metadata-place+face.png
  Expected: AI Generated
  ❌            Base:  Real (99.20%)
  ❌        GenImage:  Real (53.84%)
  ❌ GenImage then JulienLucas:  Real (90.30%)

fake+metadata-place-face.png
  Expected: AI Generated
  ❌            Base:  Real (99.01%)
  ❌        GenImage:  Real (87.42%)
  ✅ GenImage then JulienLucas:  AI Generated (75.97%)

fake-metadata+place-face.png
  Expected: AI Generated
  ❌            Base:  Real (99.06%)
  ❌        GenImage:  Real (69.15%)
  ✅ GenImage then JulienLucas:  AI Generated (66.36%)

fake-metadata-place+face.png
  Expected: AI Generated
  ❌            Base:  Real (99.20%)
  ❌        GenImage:  Real (53.84%)
  ❌ GenImage then JulienLucas:  Real (90.30%)

fake-metadata-place-face.png
  Expected: AI Generated
  ❌            Base:

---
## 📝 Quick Reference — Example Workflows

### Workflow A: GenImage first, then julienlucas
```
Run 1:  RUN_NAME="run_01_genimage",    START_FROM="base",                        TRAIN_DATASET="genimage"
Run 2:  RUN_NAME="run_02_julienlucas", START_FROM="current",                     TRAIN_DATASET="julienlucas", REPLAY_DATASET="genimage"
```

### Workflow B: julienlucas first, then genimage
```
Run 1:  RUN_NAME="run_01_julienlucas", START_FROM="base",                        TRAIN_DATASET="julienlucas"
Run 2:  RUN_NAME="run_02_genimage",    START_FROM="current",                     TRAIN_DATASET="genimage",    REPLAY_DATASET="julienlucas"
```

### Workflow C: Re-run with different hyperparams
```
Run 1:  RUN_NAME="run_01_genimage_lr2e5",  START_FROM="base",  LEARNING_RATE=2e-5
Run 2:  RUN_NAME="run_01_genimage_lr1e5",  START_FROM="base",  LEARNING_RATE=1e-5
```

### Workflow D: Resume from a saved checkpoint
```
Run 1:  RUN_NAME="run_03_continued",  START_FROM="outputs/models/run_01_genimage",  TRAIN_DATASET="julienlucas"
```